# Semana 14 · Del modelo al artefacto desplegable

> **Versión estudiante.** Ejecute en orden y conserve la salida como evidencia.

**Objetivo:** entrenar un clasificador reproducible, serializar el modelo y su contrato de inferencia, y descargar un paquete que pueda consumir Streamlit dentro de Docker.

Colab es el taller de entrenamiento, no el servidor. El despliegue persistente ocurre en la VM.

## 1. Preparar el entorno

Si está en Colab, la primera celda instala versiones compatibles. En ejecución local no modifica el entorno.

In [ ]:
import subprocess
import sys

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable,
        "-m",
        "pip",
        "install",
        "-q",
        "pandas==3.0.3",
        "numpy==2.4.6",
        "scikit-learn==1.9.0",
        "joblib==1.6.0",
    ])

## 2. Cargar datos

Suba Customer_Churn_Iran.csv cuando aparezca el selector. Si el archivo ya está junto al notebook, se usa directamente.

In [ ]:
from pathlib import Path
import json
import platform
import zipfile

import joblib
import numpy as np
import pandas as pd
import sklearn
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, balanced_accuracy_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

DATA = Path("Customer_Churn_Iran.csv")
if not DATA.exists() and IN_COLAB:
    from google.colab import files
    uploaded = files.upload()
    DATA = Path(next(iter(uploaded)))

assert DATA.exists(), "Suba Customer_Churn_Iran.csv antes de continuar."
df = pd.read_csv(DATA)
print(df.shape)
display(df.head())

## 3. Separar antes de ajustar

Se preserva un conjunto de prueba que no participa en el ajuste ni en la selección del umbral. La validación escoge el umbral que maximiza *balanced accuracy*.

In [ ]:
TARGET = "Churn"
X = df.drop(columns=TARGET)
y = df[TARGET].astype(int)

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.40, stratify=y, random_state=42
)
X_valid, X_test, y_valid, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, stratify=y_temp, random_state=42
)

model = RandomForestClassifier(
    n_estimators=300,
    min_samples_leaf=5,
    class_weight="balanced",
    random_state=42,
    n_jobs=-1,
)
model.fit(X_train, y_train)

valid_probability = model.predict_proba(X_valid)[:, 1]
thresholds = np.linspace(0.20, 0.80, 61)
scores = [balanced_accuracy_score(y_valid, valid_probability >= t) for t in thresholds]
threshold = float(thresholds[int(np.argmax(scores))])
print({"threshold": threshold, "balanced_accuracy_valid": max(scores)})

## 4. Evaluar una sola vez

Estas métricas describen este corte de prueba; no prometen desempeño futuro. El umbral pertenece al contrato de decisión.

In [ ]:
test_probability = model.predict_proba(X_test)[:, 1]
test_prediction = (test_probability >= threshold).astype(int)
metrics = {
    "balanced_accuracy_test": balanced_accuracy_score(y_test, test_prediction),
    "accuracy_test": accuracy_score(y_test, test_prediction),
    "precision_test": precision_score(y_test, test_prediction),
    "recall_test": recall_score(y_test, test_prediction),
    "roc_auc_test": roc_auc_score(y_test, test_probability),
}
display(pd.Series(metrics).round(4))

## 5. Exportar modelo, contrato y caso de referencia

El contrato conserva orden, rango, tipo y valor inicial de cada entrada. La predicción de referencia detecta errores al cambiar de entorno.

In [ ]:
ARTIFACTS = Path("artifacts")
ARTIFACTS.mkdir(exist_ok=True)

feature_order = X.columns.tolist()
input_contract = {}
for feature in feature_order:
    series = X_train[feature]
    input_contract[feature] = {
        "minimum": float(series.min()),
        "maximum": float(series.max()),
        "default": float(series.median()),
        "integer": bool(pd.api.types.is_integer_dtype(series)),
    }

importance = sorted(
    ({"feature": name, "importance": float(value)} for name, value in zip(feature_order, model.feature_importances_)),
    key=lambda item: item["importance"],
    reverse=True,
)
metadata = {
    "model_name": "RandomForestClassifier",
    "model_version": "1.0.0",
    "dataset": DATA.name,
    "target": TARGET,
    "feature_order": feature_order,
    "input_contract": input_contract,
    "decision_threshold": threshold,
    "metrics": {key: round(float(value), 6) for key, value in metrics.items()},
    "global_importance": importance,
    "runtime": {"python": platform.python_version(), "scikit_learn": sklearn.__version__},
    "limitations": [
        "Demostración académica; no usar para decisiones reales sin validación externa.",
        "La explicación SHAP describe el modelo y no implica causalidad.",
        "Los rangos reflejan el conjunto de entrenamiento y deben monitorearse en producción.",
    ],
}

joblib.dump(model, ARTIFACTS / "model.joblib")
(ARTIFACTS / "model_metadata.json").write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding="utf-8")

reference_input = X_test.iloc[0].to_dict()
reference_probability = float(model.predict_proba(pd.DataFrame([reference_input], columns=feature_order))[0, 1])
reference = {
    "input": {key: float(value) for key, value in reference_input.items()},
    "probability": reference_probability,
    "prediction": int(reference_probability >= threshold),
    "tolerance": 0.001,
}
(ARTIFACTS / "reference_prediction.json").write_text(json.dumps(reference, indent=2), encoding="utf-8")
print(sorted(path.name for path in ARTIFACTS.iterdir()))

## 6. Verificar paridad y descargar

La comprobación recarga el objeto desde disco: prueba el artefacto que se enviará, no solamente el modelo en memoria.

In [ ]:
loaded_model = joblib.load(ARTIFACTS / "model.joblib")
loaded_metadata = json.loads((ARTIFACTS / "model_metadata.json").read_text(encoding="utf-8"))
loaded_reference = json.loads((ARTIFACTS / "reference_prediction.json").read_text(encoding="utf-8"))
frame = pd.DataFrame([loaded_reference["input"]], columns=loaded_metadata["feature_order"])
actual = float(loaded_model.predict_proba(frame)[0, 1])
assert abs(actual - loaded_reference["probability"]) <= loaded_reference["tolerance"]
print("Paridad verificada:", actual)

with zipfile.ZipFile("artifacts.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(ARTIFACTS.iterdir()):
        archive.write(path, arcname=f"artifacts/{path.name}")

if IN_COLAB:
    from google.colab import files
    files.download("artifacts.zip")

## Cierre

Explique en cuatro frases: por qué se separó validación de prueba, qué contiene el contrato, qué detecta la predicción de referencia y por qué Colab no hospeda la aplicación final.